# freeze-requires-grad — faded example 2: Partial Freeze: Keep Last N Encoder Children Trainable

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `freeze-requires-grad`. Running the beacon reports progress on the `PyTorch: freeze via requires_grad=False` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: freeze via requires_grad=False` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`freeze-requires-grad`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "freeze-requires-grad"
DD_SUBTOPIC = "PyTorch: freeze via requires_grad=False"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The partial fine-tuning pattern first freezes all parameters, then selectively re-enables the last `n` children of the encoder. Because children include parameter-free modules like ReLU, the slice `list(encoder.children())[-n:]` may include non-parameter-bearing layers — these contribute nothing to the trainable list but still count toward the slice index.

## Faded exercise 2

Implement `partial_unfreeze(model, n_last)` that: (1) sets `requires_grad = False` on every parameter; (2) re-enables `requires_grad = True` on every parameter in `list(model.encoder.children())[-n_last:]`; (3) returns `[p for p in model.parameters() if p.requires_grad]`.

**Fill in:** After freezing all parameters, iterate over list(model.encoder.children())[-n_last:] and for each child module call p.requires_grad = True on every p in child.parameters().

In [ ]:
import torch as t
import torch.nn as nn

def partial_unfreeze(model: nn.Module, n_last: int):
    for p in model.parameters():
        p.requires_grad = False
    raise NotImplementedError()  # TODO: After freezing all parameters, iterate over list(model.encoder.children())[-n_last:] and for each child module call p.requires_grad = True on every p in child.parameters().
    return [p for p in model.parameters() if p.requires_grad]


def _test():
    import torch as t
    import torch.nn as nn
    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(8, 16),
                nn.ReLU(),
                nn.Linear(16, 8),
                nn.ReLU(),
                nn.Linear(8, 4),
            )
            self.fc = nn.Linear(4, 3)
        def forward(self, x):
            return self.fc(self.encoder(x))
    model = Net()
    trainable = partial_unfreeze(model, n_last=1)
    # last child is Linear(8,4) -> 2 trainable tensors
    assert len(trainable) == 2
    # first linear in encoder is frozen
    assert not model.encoder[0].weight.requires_grad
    # last linear in encoder is trainable
    assert model.encoder[4].weight.requires_grad
    # head is frozen (we only touched encoder children)
    assert not model.fc.weight.requires_grad


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

def partial_unfreeze(model: nn.Module, n_last: int):
    for p in model.parameters():
        p.requires_grad = False
    children = list(model.encoder.children())
    for layer in children[-n_last:]:
        for p in layer.parameters():
            p.requires_grad = True
    return [p for p in model.parameters() if p.requires_grad]
```
</details>